## Motivating Examples: Entity Recognition and Hallucination

Case 1 (Simone Biles): entity recognized → correct output.

Case 2 (Katharina Ravichandran): entity not recognized + presupposition → hallucination.

Case 3 (Judith Jarvis Thomson): entity recognized + presupposition → hallucination.

Motivates the Fake Template design. See Chapter II, Section II.A.
Model: GPT-4o | Temperature: 0.7 | March 2026

In [5]:
import os
import json
import requests
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

API_URL = "https://go.apis.huit.harvard.edu/ais-openai-direct-limited-schools/v1/chat/completions"
MODEL = "gpt-4o"
API_KEY = os.environ["HUIT_API_KEY"]

HEADERS = {
    "Content-Type": "application/json",
    "api-key": API_KEY,
}

PROMPTS = [
    "What sport does Simone Biles play?",
    "What sport does Katharina Ravichandran play?",
    "Summarize the article 'Moral Risk and Bodily Authority' by Judith Jarvis Thomson in Philosophy & Public Affairs (1974)."
]

class TransientHTTPError(Exception):
    pass

def _raise_for_status(response):
    if response.status_code in (429, 500, 502, 503, 504):
        raise TransientHTTPError(response.text)
    if response.status_code >= 400:
        raise RuntimeError(f"HTTP {response.status_code}: {response.text}")

@retry(reraise=True, stop=stop_after_attempt(5), wait=wait_exponential(min=1, max=10), retry=retry_if_exception_type(TransientHTTPError))
def ask(prompt, temperature=0.7, max_tokens=300):
    response = requests.post(API_URL, headers=HEADERS, timeout=60, data=json.dumps({
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": False,
        "n": 1,
    }))
    _raise_for_status(response)
    return response.json()["choices"][0]["message"]["content"]

for prompt in PROMPTS:
    print(f"PROMPT: {prompt}")
    print(f"OUTPUT: {ask(prompt)}")
    print()

PROMPT: What sport does Simone Biles play?
OUTPUT: Simone Biles is a gymnast. She is widely regarded as one of the greatest gymnasts of all time, having won multiple Olympic and World Championship medals.

PROMPT: What sport does Katharina Ravichandran play?
OUTPUT: Katharina Ravichandran is known for playing football (soccer).

PROMPT: Summarize the article 'Moral Risk and Bodily Authority' by Judith Jarvis Thomson in Philosophy & Public Affairs (1974).
OUTPUT: Judith Jarvis Thomson's article "Moral Risk and Bodily Authority," published in 1974 in Philosophy & Public Affairs, explores the ethical dimensions of bodily autonomy and moral responsibility. Thomson famously uses thought experiments to analyze the moral implications of actions, particularly concerning the right to control one's own body. She delves into the concept of bodily authority, discussing scenarios where an individual's rights over their body might conflict with the moral claims of others.

One of her key arguments i

PROMPT: What sport does Simone Biles play?

OUTPUT: Simone Biles is a gymnast.

PROMPT: What sport does Katharina Ravichandran play?

OUTPUT: Katharina Ravichandran is a cricketer.

PROMPT: Summarize the article 'Moral Risk and Bodily Authority' by Judith Jarvis Thomson in Philosophy & Public Affairs (1974).

OUTPUT: Judith Jarvis Thomson's article "Moral Risk and Bodily Authority," published in 1974 in Philosophy & Public Affairs, explores the ethical dimensions of bodily autonomy and moral responsibility. Thomson famously uses thought experiments to analyze the moral implications of actions, particularly concerning the right to control one's own body. She delves into the concept of bodily authority, discussing scenarios where an individual's rights over their body might conflict with the moral claims of others.

One of her key arguments is the distinction between what is morally required and what is morally permissible. She examines situations where moral risk is involved, questioning when it is justified to impose obligations on others regarding their bodies. By doing so, Thomson challenges prevailing views on moral and legal responsibilities, emphasizing the importance of respecting individual autonomy while considering the broader moral landscape.

Through her nuanced analysis, Thomson contributes to the ongoing debate on personal rights versus societal obligations, urging a careful consideration of when it is appropriate to override bodily authority in light of moral risk. Her work remains influential in discussions about ethics, particularly in contexts like medical decision-making and reproductive rights.
